# DermaMNIST Full Validation — Associative Memory

Primary notebook for the **corrected** associative optimization-history protocol on DermaMNIST.

## What this notebook builds

| Path | Mechanism | Used for |
|------|-----------|----------|
| **Primary (Part I + §D.1 + probe notebook)** | Frozen per-level responses \(z_j(x)=M^j k_{\mathrm{query}}(x)\) | Paper metrics: `z_combined` vs \(H\) |
| **Optional (§3–§8, `RUN_EVALUATION=True`)** | Full checkpoint trajectory \(R\), optional rFFT, mean-pool → \(z_{\mathrm{memory}}\) | Legacy deferral / memory-ID / NRO ablations |

Training and deployment **do not apply FFT**. `ASSOC_CFG.use_fft=True` affects only the optional §3–§8 trajectory feature path (when `use_fft=False`, use raw \(R\)).

**Paper failure-detection evaluation** lives in `memory_vector_vs_magnitude_probe.ipynb` (uses artifacts from Part I here).

## Quick start — deploy only (no retraining)

1. Run **§0 Setup** (imports)
2. Run **Configuration** — defaults: `RUN_BUILD=False`, `RUN_DEPLOYMENT=True`, `RUN_EVALUATION=False`
3. Run **Shared dataset bundle**
4. Run **Part II → §D.1** (deployment smoke test)

Skip **Part I** (§1 train, §2 reconstruction) unless artifacts are missing. Skip **§3–§8** unless you need legacy downstream eval.

## Pipeline overview

| Part | Purpose | When to run |
|------|---------|-------------|
| **I — Build** | Train ResNet-18 + delta-rule memory \(\{M^j\}\); reconstruction gate | Once (`RUN_BUILD=True`) |
| **II — Deploy (§D.1)** | Label-free \(x \to h_T \to k_{\mathrm{query}} \to z_j=M^j k\); optional history \(z_{t,j}\) | Default (`RUN_DEPLOYMENT=True`) |
| **II — Eval (§3–§8)** | Memory ID, NRO, deferral, FFT ablations, verdict | Optional (`RUN_EVALUATION=True`) |

Flow diagrams: `publication/images/associative_memory_training_flow.png` and `associative_memory_deployment_flow.png`.

## 0. Setup

Import experiment drivers and put the repo on `sys.path`. No training happens here.

**Notation (primary path):**
- \(h_T(x)\): frozen penultimate ResNet-18 features after training
- \(k_{\mathrm{query}}(x)=h_T(x)/(\|h_T(x)\|+\varepsilon)\): deployment query key (no labels)
- \(M^j\): final associative matrix at level \(j\) (frozen after Part I)
- \(z_j(x)=M^j k_{\mathrm{query}}(x)\): per-level memory response used by the probe notebook
- \(\tilde H(x)\): normalized predictive entropy

**Notation (optional §3–§8 only):**
- \(h(x)\): projected penultimate features (fixed linear map)
- \(R[t,j,:]=M_t^j k_{\mathrm{query}}\): checkpoint trajectory; rFFT → \(z_{\mathrm{memory}}\) when `use_fft=True`

In [1]:
# §0 — Imports and repo path (no equations executed here).
# Primary path: train → frozen M^j → deploy z_j = M^j k_query (no FFT).
# Optional §3–§8: trajectory R, optional rFFT → z_memory (ASSOC_CFG.use_fft).

from __future__ import annotations

import json
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd

def find_repo_root(cwd: Path | None = None) -> Path:
    """Locate repo root so `from research...` imports work from notebooks/."""
    root = Path(cwd or Path.cwd()).resolve()
    if (root / "research").exists():
        return root
    if (root.parent / "research").exists():
        return root.parent
    return root

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# AssociativeMemoryConfig: use_fft / use_attention affect §3–§8 z_memory only (attention off here)
from optimizer.associative_memory import AssociativeMemoryConfig
from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import (
    ClinicalTrainingConfig,
    checkpoint_path,
    export_scoring_artifacts_for_nro,
    save_checkpoint,
    save_frozen_memory,
    train_one_seed,
)
from research.common.memory import (
    associative_artifacts_exist,
    associative_memory_path,
    checkpoint_bundle_path,
    load_associative_artifacts,
)
from research.common.deployment_pipeline import (
    offline_evaluate_memory_information,
    plot_deployment_diagnostics,
    run_deployment_on_split,
    run_deployment_sanity_checks,
    save_deployment_records,
    summarize_deployment_diagnostics,
)
from research.common.msa_linear_deferral import MSALinearDeferralConfig, run_msa_linear_deferral_experiment
from research.incremental_memory_deferral.incremental_memory_deferral import (
    IncrementalDeferralConfig,
    run_incremental_memory_deferral,
)
from research.memory_identification.memory_identification import MemoryIdentificationConfig, run_memory_identification
from research.common.nro_final_experiment import NROFinalConfig, run_nro_final_experiment

print("imports ok")

imports ok


## Configuration

Sets paths, phase switches, and `ASSOC_CFG`.

### Phase switches (edit these)

| Flag | Default (deploy-only) | Meaning |
|------|----------------------|---------|
| `RUN_BUILD` | `False` | Part I: train + reconstruction gate |
| `RUN_DEPLOYMENT` | `True` | Part II §D.1: label-free deployment pipeline |
| `RUN_EVALUATION` | `False` | Part II §3–§8: memory-ID, NRO, deferral, FFT ablations |

**Train/deploy (primary):** moving keys \(k_t\) and labels \(v=y-p\) only during Part I. Deployment queries frozen \(M^j\) with \(k_{\mathrm{query}}\) — **no FFT, no \(v\)**.

**`ASSOC_CFG`:** `AssociativeMemoryConfig(use_fft=True, use_attention=False)`. FFT/attention flags apply to **§3–§8 trajectory features only**, not to §1 training or §D.1 deployment.

Artifacts under `OUTPUT_DIR/memory/`: `*_associative.npz` (final \(M^j\)), `*_checkpoints.npz` (history grid).

In [2]:
# Paths, seeds, phase switches, and associative-memory toggles.
# ASSOC_CFG.use_fft / use_attention → §3–§8 trajectory features only (not train or §D.1).

TASK = "dermamnist"
OUTPUT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "artifacts"
DATA_DIR = REPO_ROOT / "data" / "clinical"
SEEDS = (42, 123, 456)

# --- Phase switches (deploy-only: BUILD off, DEPLOYMENT on) ---
RUN_BUILD = False
RUN_DEPLOYMENT = True
RUN_EVALUATION = False  # True → §3–§8 legacy downstream (memory-ID, NRO, deferral, ablations)

FAST_MODE = False
FORCE_RETRAIN = False
FORCE_RERUN_MEMORY_ID = False

# Deployment (Part II) — uses frozen checkpoints from OUTPUT_DIR
DEPLOY_SEEDS = SEEDS
DEPLOY_SPLITS = ("cal", "external")
DEPLOYMENT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "deployment" / TASK
DEPLOY_MAX_SAMPLES = 100  # e.g. 64 for a quick smoke deploy
DEPLOY_INCLUDE_HISTORY = True

if FAST_MODE:
    SEEDS = (42,)
    DEPLOY_SEEDS = (42,)
    EPOCHS, MAX_TRAIN, MAX_CAL, MAX_TEST, MAX_EXT = 2, 800, 200, 400, 400
    DEPLOY_MAX_SAMPLES = DEPLOY_MAX_SAMPLES or 200
else:
    EPOCHS, MAX_TRAIN, MAX_CAL, MAX_TEST, MAX_EXT = 8, None, None, None, None

ASSOC_CFG = AssociativeMemoryConfig(use_fft=True, use_attention=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR", OUTPUT_DIR)
print("DEPLOYMENT_DIR", DEPLOYMENT_DIR)
print(f"RUN_BUILD={RUN_BUILD}  RUN_DEPLOYMENT={RUN_DEPLOYMENT}  RUN_EVALUATION={RUN_EVALUATION}")

OUTPUT_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\artifacts
DEPLOYMENT_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\deployment\dermamnist
RUN_BUILD=False  RUN_DEPLOYMENT=True  RUN_EVALUATION=True


In [3]:
# Shared dataset bundle (required for Build and Deployment)

ds_cfg = ClinicalDatasetConfig(
    task=TASK,
    data_dir=DATA_DIR,
    max_train=MAX_TRAIN,
    max_cal=MAX_CAL,
    max_test=MAX_TEST,
    max_external=MAX_EXT,
)
bundle = load_clinical_bundle(ds_cfg)
print(
    f"bundle: task={bundle.task} classes={bundle.num_classes} "
    f"n_cal={len(bundle.x_cal)} n_external={len(bundle.x_external)}"
)

bundle: task=dermamnist classes=7 n_cal=1602 n_external=2005


---

# Part I — Build (Training)

Trains ResNet-18 (Adam) in parallel with a **delta-rule associative side channel** (four levels \(M^j\)). No FFT in this phase — only matrix updates and checkpoint saves.

Skip when `RUN_BUILD=False`. Uses cached checkpoints if they exist (unless `FORCE_RETRAIN=True`).

| Step | Output |
|------|--------|
| §1 Train | `checkpoints/{task}_seed{seed}.pkl`, `memory/*_associative.npz`, `memory/*_checkpoints.npz` |
| §2 Reconstruction | `memory_reconstruction_diagnostics.csv` (cosine/MSE of \(M^j k\) vs \(v=y-p\)) |

After Part I, run **`memory_vector_vs_magnitude_probe.ipynb`** for paper AUROC/AUPRC (`z_combined` vs \(\tilde H\)).

## §1 Train (associative memory side-channel)

Runs only when `RUN_BUILD=True`. Each batch: classifier Adam step + optional \(M^j\) delta updates with \(k_t=\mathrm{normalize}(h_t)\), \(v_t=y_t-p_t\) (labels train-only).

~101 batches/epoch, 8 epochs, 3 seeds — expect ~1 h/seed on CPU. No FFT in training.

In [3]:
# §1 — Train backbone + associative memory (Part I only)

summaries = []

def remove_training_artifacts(output_dir: Path, task: str, seed: int) -> None:
    for path in (
        checkpoint_path(output_dir, task, seed),
        output_dir / "memory" / f"{task}_seed{seed}_memory.npz",
        associative_memory_path(output_dir, task, seed),
        checkpoint_bundle_path(output_dir, task, seed),
        output_dir / "calibration" / f"{task}_seed{seed}.csv",
        output_dir / "scored" / f"{task}_seed{seed}.csv",
    ):
        if path.exists():
            path.unlink()
            print(f"removed {path.name}")

if not RUN_BUILD:
    print("RUN_BUILD=False — skipping training (deploy-only mode)")
    missing = [s for s in SEEDS if not associative_artifacts_exist(OUTPUT_DIR, TASK, s)]
    if missing:
        raise FileNotFoundError(
            f"Missing associative artifacts for seeds {missing}. "
            "Run Part I with RUN_BUILD=True or restore OUTPUT_DIR/memory/*."
        )
else:
    train_cfg = ClinicalTrainingConfig(
        seeds=SEEDS,
        epochs=EPOCHS,
        output_dir=OUTPUT_DIR,
        verbose=1,
        associative_memory=ASSOC_CFG,
    )
    for seed in SEEDS:
        ckpt = checkpoint_path(OUTPUT_DIR, TASK, seed)
        if FORCE_RETRAIN:
            print(f"force retrain seed={seed}")
            remove_training_artifacts(OUTPUT_DIR, TASK, seed)
        elif ckpt.exists() and associative_artifacts_exist(OUTPUT_DIR, TASK, seed):
            print(f"skip training seed={seed}, checkpoint and associative artifacts exist")
            continue
        elif ckpt.exists():
            print(f"retraining seed={seed}: model checkpoint exists but associative artifacts are missing")
        params, opt_state, per_sample, metrics, summary = train_one_seed(bundle, seed=seed, cfg=train_cfg)
        save_checkpoint(ckpt, params=params, opt_state=opt_state, task=TASK, seed=seed, cfg=train_cfg, summary=summary)
        save_frozen_memory(OUTPUT_DIR / "memory" / f"{TASK}_seed{seed}_memory.npz", opt_state, task=TASK, seed=seed)
        export_scoring_artifacts_for_nro(
            bundle, params=params, opt_state=opt_state, seed=seed, cfg=train_cfg, output_dir=OUTPUT_DIR
        )
        summaries.append(summary)
        print(seed, summary.get("test_acc"), summary.get("checkpoint_count"))


force retrain seed=42
removed dermamnist_seed42.pkl
removed dermamnist_seed42_memory.npz
removed dermamnist_seed42_associative.npz
removed dermamnist_seed42_checkpoints.npz
removed dermamnist_seed42.csv
removed dermamnist_seed42.csv
  [dermamnist seed=42] training: 6408 samples, 8 epochs, ~101 batches/epoch
  [dermamnist seed=42] epoch 1/8: train_loss=0.9635 cal_acc=0.660 test_acc=0.660 (610.6s)
  [dermamnist seed=42] epoch 2/8: train_loss=0.8569 cal_acc=0.667 test_acc=0.677 (543.3s)
  [dermamnist seed=42] epoch 3/8: train_loss=0.8149 cal_acc=0.701 test_acc=0.697 (439.7s)
  [dermamnist seed=42] epoch 4/8: train_loss=0.7807 cal_acc=0.710 test_acc=0.697 (428.7s)
  [dermamnist seed=42] epoch 5/8: train_loss=0.7521 cal_acc=0.714 test_acc=0.711 (430.0s)
  [dermamnist seed=42] epoch 6/8: train_loss=0.7237 cal_acc=0.723 test_acc=0.738 (432.0s)
  [dermamnist seed=42] epoch 7/8: train_loss=0.6954 cal_acc=0.730 test_acc=0.718 (438.1s)
  [dermamnist seed=42] epoch 8/8: train_loss=0.6722 cal_acc=0

## §2 Memory reconstruction diagnostics (Build gate)

Checks whether frozen \(M^j k\) aligns with training residuals \(v=y-p\) on a probe set. Moderate cosine (~0.2–0.3) is expected; imperfect reconstruction does not block deployment.

Runs when `RUN_BUILD=True`; otherwise loads cached `memory_reconstruction_diagnostics.csv` if present.

In [5]:
# §2 — Reconstruction diagnostics (Part I only; labels used here only)

import pickle
import jax.numpy as jnp
import numpy as np
from optimizer.associative_memory import value_from_logits
from research.common.resnet import resnet18_features, resnet18_apply
from research.common.memory import load_associative_artifacts, memory_reconstruction_diagnostics

recon_csv = OUTPUT_DIR / "memory_reconstruction_diagnostics.csv"

if RUN_BUILD:
    diag_frames = []
    for seed in SEEDS:
        state, checkpoints, _cfg = load_associative_artifacts(OUTPUT_DIR, TASK, seed)
        with open(checkpoint_path(OUTPUT_DIR, TASK, seed), "rb") as f:
            params = pickle.load(f)["params"]
        x = bundle.x_train[: min(64, len(bundle.x_train))]
        y = bundle.y_train[: len(x)]
        x_j = jnp.asarray(x, dtype=jnp.float32)
        logits = resnet18_apply(params, x_j)
        h = np.asarray(resnet18_features(params, x_j))
        keys = h / (np.linalg.norm(h, axis=-1, keepdims=True) + 1e-8)
        values = np.asarray(value_from_logits(logits, jnp.asarray(y), bundle.num_classes))
        diag = memory_reconstruction_diagnostics(state, keys, values)
        diag["seed"] = seed
        diag_frames.append(diag)
    recon_df = pd.concat(diag_frames, ignore_index=True)
    recon_summary = recon_df.groupby("level")[["mse", "cosine"]].mean()
    display(recon_summary)
    recon_df.to_csv(recon_csv, index=False)
elif recon_csv.exists():
    recon_df = pd.read_csv(recon_csv)
    recon_summary = recon_df.groupby("level")[["mse", "cosine"]].mean()
    print(f"loaded cached reconstruction from {recon_csv.name}")
    display(recon_summary)
else:
    recon_df = None
    recon_summary = None
    print("RUN_BUILD=False and no cached reconstruction CSV — skipping §2")


,mse,cosine
level,,
1,0.040884,0.312841
2,0.040781,0.220425
3,0.040741,0.220155
4,0.040753,0.179921


---

# Part II — Deployment (Inference)

**Primary label-free path** (matches publication deployment diagram):

\[
x \to h_T(x) \to k_{\mathrm{query}}(x)=\frac{h_T}{\|h_T\|+\varepsilon} \to z_j(x)=M^j k_{\mathrm{query}}(x)
\]

Optional: historical checkpoints \(z_{t,j}(x)=M_t^j k_{\mathrm{query}}(x)\) when `DEPLOY_INCLUDE_HISTORY=True`.

**No FFT. No \(v=y-p\).** Labels joined only for offline error/AUROC checks. Full probe metrics: `memory_vector_vs_magnitude_probe.ipynb`.

Run when `RUN_DEPLOYMENT=True` (default). Uses frozen \(\theta_T\) and \(\{M^j\}\) from `OUTPUT_DIR`.

## §D.1 Label-free deployment + offline memory evaluation

Smoke-test deployment (not the full paper eval table). Per seed:

1. Sanity checks on frozen artifacts
2. Deploy on `DEPLOY_SPLITS` (default: `cal`, `external`; subsample via `DEPLOY_MAX_SAMPLES`)
3. Write JSONL/CSV under `DEPLOYMENT_DIR/{seed}/`
4. Quick logistic AUROC: \(H\) vs memory magnitudes vs combined

For publication numbers (`z_combined` vs \(\tilde H\), AUROC/AUPRC, risk–coverage), use **`memory_vector_vs_magnitude_probe.ipynb`** on full cal/test splits.

In [4]:
# §D.1 — Deployment pipeline (inference only; no training)

import pickle

split_data = {
    "cal": (bundle.x_cal, bundle.y_cal, bundle.sample_ids["cal"]),
    "test": (bundle.x_test, bundle.y_test, bundle.sample_ids["test"]),
    "external": (bundle.x_external, bundle.y_external, bundle.sample_ids["external"]),
}

deployment_outputs: dict[int, dict[str, pd.DataFrame]] = {}

if not RUN_DEPLOYMENT:
    print("RUN_DEPLOYMENT=False — skipping Part II deployment")
else:
    for seed in DEPLOY_SEEDS:
        if not associative_artifacts_exist(OUTPUT_DIR, TASK, seed):
            raise FileNotFoundError(
                f"Missing artifacts for seed={seed}. Run Part I or restore OUTPUT_DIR/memory/*."
            )
        ckpt = checkpoint_path(OUTPUT_DIR, TASK, seed)
        with open(ckpt, "rb") as f:
            params = pickle.load(f)["params"]
        mem_state, checkpoints, _assoc_cfg = load_associative_artifacts(OUTPUT_DIR, TASK, seed)

        seed_dir = DEPLOYMENT_DIR / f"seed{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)

        x_sanity = split_data["cal"][0][: min(4, len(split_data["cal"][0]))]
        sanity = run_deployment_sanity_checks(
            params,
            x_sanity,
            mem_state,
            num_classes=bundle.num_classes,
            checkpoints=checkpoints if DEPLOY_INCLUDE_HISTORY else None,
        )
        with open(seed_dir / "sanity_report.json", "w", encoding="utf-8") as f:
            json.dump(
                {"passed": sanity.passed, "checks": sanity.checks, "details": sanity.details},
                f,
                indent=2,
                default=str,
            )
        print(f"seed={seed} sanity passed={sanity.passed}")

        seed_frames: dict[str, pd.DataFrame] = {}
        history_for_plots = []
        for split in DEPLOY_SPLITS:
            x, y, ids = split_data[split]
            if DEPLOY_MAX_SAMPLES is not None:
                x, y, ids = x[:DEPLOY_MAX_SAMPLES], y[:DEPLOY_MAX_SAMPLES], ids[:DEPLOY_MAX_SAMPLES]
            records, df = run_deployment_on_split(
                params,
                x,
                y,
                ids,
                mem_state,
                checkpoints,
                num_classes=bundle.num_classes,
                include_history=DEPLOY_INCLUDE_HISTORY,
            )
            if DEPLOY_INCLUDE_HISTORY and not history_for_plots:
                history_for_plots = records[: min(8, len(records))]
            paths = save_deployment_records(records, seed_dir / split, prefix=f"{split}_deployment")
            summary = summarize_deployment_diagnostics(df)
            summary.to_csv(seed_dir / split / f"{split}_diagnostic_summary.csv", index=False)
            seed_frames[split] = df
            print(f"  {split}: {len(records)} samples → {paths['csv'].name}")

        if "cal" in seed_frames and "external" in seed_frames:
            offline = offline_evaluate_memory_information(seed_frames["cal"], seed_frames["external"])
            offline.to_csv(seed_dir / "offline_evaluation.csv", index=False)
            display(offline)

        if "external" in seed_frames:
            plot_deployment_diagnostics(
                seed_frames["external"],
                seed_dir / "plots",
                prefix="external",
                history_records=history_for_plots if DEPLOY_INCLUDE_HISTORY else None,
            )

        deployment_outputs[seed] = seed_frames

    print("deployment complete →", DEPLOYMENT_DIR)

seed=42 sanity passed=True
  cal: 100 samples → cal_deployment_summary.csv
  external: 100 samples → external_deployment_summary.csv


,model,auroc,auprc,n_train,n_test
0,prediction_only_H,0.635570,0.425567,100.0,100
1,memory_only,0.574449,0.438026,100.0,100
2,combined_H_and_memory,0.598346,0.480166,100.0,100
3,univariate_normalized_entropy,0.635570,0.425567,NaN,100
4,univariate_cross_level_mean_cosine,0.628676,0.484075,NaN,100
5,univariate_cross_level_magnitude_variance,0.490809,0.301056,NaN,100
6,univariate_cross_level_response_variance_mean,0.482537,0.299260,NaN,100
7,univariate_z1_magnitude,0.483456,0.297523,NaN,100
8,univariate_z2_magnitude,0.596967,0.410541,NaN,100
9,univariate_z3_magnitude,0.596507,0.416431,NaN,100


seed=123 sanity passed=True
  cal: 100 samples → cal_deployment_summary.csv
  external: 100 samples → external_deployment_summary.csv


,model,auroc,auprc,n_train,n_test
0,prediction_only_H,0.630055,0.444376,100.0,100
1,memory_only,0.653033,0.457376,100.0,100
2,combined_H_and_memory,0.649357,0.470835,100.0,100
3,univariate_normalized_entropy,0.630055,0.444376,NaN,100
4,univariate_cross_level_mean_cosine,0.495404,0.356159,NaN,100
5,univariate_cross_level_magnitude_variance,0.662684,0.478162,NaN,100
6,univariate_cross_level_response_variance_mean,0.664982,0.482565,NaN,100
7,univariate_z1_magnitude,0.654412,0.474059,NaN,100
8,univariate_z2_magnitude,0.736673,0.574088,NaN,100
9,univariate_z3_magnitude,0.358456,0.256117,NaN,100


seed=456 sanity passed=True
  cal: 100 samples → cal_deployment_summary.csv
  external: 100 samples → external_deployment_summary.csv


,model,auroc,auprc,n_train,n_test
0,prediction_only_H,0.456801,0.313017,100.0,100
1,memory_only,0.661305,0.503393,100.0,100
2,combined_H_and_memory,0.645680,0.480843,100.0,100
3,univariate_normalized_entropy,0.456801,0.313017,NaN,100
4,univariate_cross_level_mean_cosine,0.301930,0.240292,NaN,100
5,univariate_cross_level_magnitude_variance,0.534007,0.389418,NaN,100
6,univariate_cross_level_response_variance_mean,0.580882,0.424661,NaN,100
7,univariate_z1_magnitude,0.556066,0.409917,NaN,100
8,univariate_z2_magnitude,0.750460,0.584185,NaN,100
9,univariate_z3_magnitude,0.446691,0.339088,NaN,100


deployment complete → C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\deployment\dermamnist


---

## Part II — Downstream evaluation (optional, legacy protocol)

Sections **§3–§8** use the **trajectory + optional FFT** feature path (`z_memory`), not the primary frozen \(z_j=M^j k\) probe used in the paper.

| Section | Feature path | Paper equivalent |
|---------|--------------|------------------|
| §3–§8 (here) | \(R \to\) rFFT (if on) \(\to z_{\mathrm{memory}}\) | Legacy deferral / NRO ablations |
| `memory_vector_vs_magnitude_probe.ipynb` | \(z_j=M^j k_{\mathrm{query}}\) → `z_combined` | **Primary AUROC/AUPRC** |

**Skipped** when `RUN_EVALUATION=False` (default). Set `RUN_EVALUATION=True` only if you need memory-ID, NRO, deferral, or FFT ablations in this notebook.

## §3 Failure detection (memory identification)

**Optional legacy eval** — uses trajectory features \(z_{\mathrm{memory}}\), not the primary \(z_j\) probe.

Tests whether \(z_{\mathrm{memory}}\) adds incremental failure signal beyond \(\tilde H\) and projected \(h(x)\).

**Feature construction (§3 only — label-free deploy, then join labels for AUROC):**

\[
R[\ell,j,:] = M_{t_\ell}^j k_{\mathrm{query}}, \quad
S = |\mathrm{rFFT}_t(R)| \;\text{(when `ASSOC_CFG.use_fft=True`)}, \quad
z_{\mathrm{memory}} = W \, \mathrm{mean}_{\ell,j}(S)
\]

When `use_fft=False`, use raw \(R\) instead of \(S\).

**Experiment 1 — conditional information:** logistic models on calibration:

\[
\mathbb{P}(\text{error}\mid \cdot): \quad
[\tilde H, h] \quad\text{vs}\quad [\tilde H, h, z_{\mathrm{memory}}]
\]

**Experiment 2 — ablation ladder:** subsets \(\{\tilde H\}, \{\tilde H,h\}, \{\tilde H,h,z\}\). Evidence class A/B/C from pooled tests.

For paper metrics on \(z_{\mathrm{combined}}\) vs \(\tilde H\), use **`memory_vector_vs_magnitude_probe.ipynb`**.

In [5]:
# §3 — Legacy memory-ID on trajectory z_memory (optional; RUN_EVALUATION=True)

import shutil

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §3 memory identification")
    mem_id_results = None
else:
    _mem_id_dir = OUTPUT_DIR / "memory_identification"
    if FORCE_RERUN_MEMORY_ID and _mem_id_dir.exists():
        shutil.rmtree(_mem_id_dir)
        print(f"removed cached memory-identification results in {_mem_id_dir.name}")

    _exp1_cache = _mem_id_dir / "experiment1_per_seed.csv"
    _run_exp1 = True
    if _exp1_cache.exists():
        _cached_seeds = set(pd.read_csv(_exp1_cache)["seed"].astype(int))
        if set(SEEDS).issubset(_cached_seeds):
            _run_exp1 = False
            print(f"exp1 already complete for seeds {SEEDS}; resuming at exp2")

    mem_id_cfg = MemoryIdentificationConfig(
        repo_root=REPO_ROOT,
        data_dir=DATA_DIR,
        source_experiment_dir=OUTPUT_DIR,
        output_dir=_mem_id_dir,
        task=TASK,
        seeds=SEEDS,
        use_associative_memory=True,
        associative_memory=ASSOC_CFG,
        train_if_checkpoint_missing=False,
        run_experiment1=_run_exp1,
    )
    mem_id_results = run_memory_identification(mem_id_cfg)
    mem_id_results.keys()


exp1 already complete for seeds (42, 123, 456); resuming at exp2
[exp1] loaded cached results (51 rows)
[exp2] loaded cached seeds [42, 123, 456]


## §4 Conditional information (NRO)

**Optional legacy eval** on trajectory feature \(z_{\mathrm{memory}}\) (FFT path when enabled).

**Nested Residual Optimization (NRO)** tests whether memory features explain residual failure risk after controlling for base scores.

Fit nested logistic models on calibration, evaluate on test/external:

\[
\text{outer: } \mathbb{P}(\text{error}\mid \tilde H, h, z_{\mathrm{memory}}), \qquad
\text{inner: } \mathbb{P}(\text{error}\mid \tilde H, h)
\]

Likelihood-ratio / information gain: does adding \(z_{\mathrm{memory}}\) significantly improve fit? Decision gate maps to proceed / caution / stop.

In [ ]:
# §4 — NRO final experiment

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §4 NRO")
    nro_results = None
else:
    nro_cfg = NROFinalConfig(
        repo_root=REPO_ROOT,
        source_experiment_dir=OUTPUT_DIR,
        output_dir=OUTPUT_DIR / "nro_final",
        seeds=SEEDS,
        task=TASK,
    )
    nro_results = run_nro_final_experiment(nro_cfg)
    nro_results.keys()


## §5 Deferral (\(\tilde H\), \(\tilde H+h\), \(\tilde H+h+z_{\mathrm{memory}}\))

**Optional legacy eval** — deferral scorers use trajectory \(z_{\mathrm{memory}}\) (with optional rFFT), not primary \(z_j\).

**Selective prediction:** defer highest-risk cases; measure error rate on the non-deferred subset.

Three deferral scorers (logistic on calibration):

| Model | Features | Score |
|-------|----------|-------|
| H | normalized entropy only | \(\hat{p}(\text{error}\mid \tilde H)\) |
| H+h | entropy + projected features | \(\hat{p}(\text{error}\mid \tilde H, h)\) |
| H+h+z | + trajectory memory vector | \(\hat{p}(\text{error}\mid \tilde H, h, z_{\mathrm{memory}})\) |

**Selective risk at deferral rate \(\alpha\):** sort by defer score, defer top \(\alpha\) fraction, report error rate on remainder.

Paper risk–coverage for `z_combined` vs \(\tilde H\): **`memory_vector_vs_magnitude_probe.ipynb`**.

\[
\text{SR}(\alpha) = \frac{1}{|\mathcal{S}_\alpha|}\sum_{i \notin \mathcal{D}_\alpha} \mathbf{1}[\hat{y}_i \neq y_i]
\]

Primary metric: improvement of H+h+z over H+h at \(\alpha = 0.20\).

In [ ]:
# §5 — Legacy deferral on trajectory z_memory (optional; RUN_EVALUATION=True)

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §5 deferral")
    defer_cfg = None
    defer_results = None
else:
    defer_cfg = MSALinearDeferralConfig(
        repo_root=REPO_ROOT,
        data_dir=DATA_DIR,
        source_experiment_dir=OUTPUT_DIR,
        output_dir=OUTPUT_DIR / "deferral",
        seeds=SEEDS,
        task=TASK,
        use_associative_memory=True,
        associative_memory=ASSOC_CFG,
        max_train=MAX_TRAIN,
        max_cal=MAX_CAL,
        max_test=MAX_TEST,
        max_external=MAX_EXT,
    ).resolve_paths()
    defer_results = run_msa_linear_deferral_experiment(defer_cfg)
    defer_results["pooled_metrics"]


C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral.py:393: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cal_df[f"z_{j}"] = z[:, j]
C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral.py:662: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cal_df[f"z_{j}"] = z_cal[:, j]
C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral

{'H': {'n': 6015,
  'deferral_rate': 0.19351620947630924,
  'coverage': 0.8064837905236908,
  'selective_risk': 0.18759018759018758,
  'selective_accuracy': 0.8124098124098125,
  'error_rate_deferred': 0.6039518900343642,
  'deferral_precision': 0.6039518900343642,
  'error_capture': 0.4358338499690019,
  'n_deferred': 1164,
  'n_errors': 1613,
  'n_errors_deferred': 703,
  'risk_all': 0.2681629260182876,
  'risk_reduction': 0.08057273842810003,
  'auroc': 0.8208791979523482,
  'auprc': 0.5774551997484805,
  'aurc': 0.08493612732422019,
  'scorer': 'H',
  'seed': 'pooled'},
 'N': {'n': 6015,
  'deferral_rate': 0.19035743973399832,
  'coverage': 0.8096425602660017,
  'selective_risk': 0.20390143737166325,
  'selective_accuracy': 0.7960985626283368,
  'error_rate_deferred': 0.5414847161572053,
  'deferral_precision': 0.5414847161572053,
  'error_capture': 0.384376937383757,
  'n_deferred': 1145,
  'n_errors': 1613,
  'n_errors_deferred': 620,
  'risk_all': 0.2681629260182876,
  'risk_red

## §6 Incremental deferral

**Optional legacy eval** — paired bootstrap on trajectory-based deferral (§5).

Does adding \(z_{\mathrm{memory}}\) incrementally improve selective risk over \(\tilde H+h\)?

\[
\Delta \text{SR}(\alpha) = \text{SR}_{H+h}(\alpha) - \text{SR}_{H+h+z}(\alpha)
\]

Positive \(\Delta\)SR means memory reduces error on the kept (non-deferred) set. Reports per-seed and pooled CIs at target deferral rates.

In [ ]:
# §6 — Paired bootstrap CI for ΔSR = SR(H+h) - SR(H+h+z) at each deferral rate.

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §6 incremental deferral")
    incr_results = None
else:
    incr_cfg = IncrementalDeferralConfig(
        mha=replace(defer_cfg, output_dir=OUTPUT_DIR / "incremental_deferral"),
    ).resolve()
    incr_results = run_incremental_memory_deferral(incr_cfg)
    incr_results["paired_ci"]


,comparison,delta_selective_risk_mean,delta_selective_risk_ci_low,delta_selective_risk_ci_high,delta_selective_risk_ci_excludes_zero,delta_deferral_precision_mean,delta_deferral_precision_ci_low,delta_deferral_precision_ci_high,delta_deferral_precision_ci_excludes_zero,delta_error_capture_mean,delta_error_capture_ci_low,delta_error_capture_ci_high,delta_error_capture_ci_excludes_zero,scope
0,per_seed_H+h_vs_H+h+z_actual,-0.000112,-0.002439,0.001895,False,-0.001170,-0.012462,0.007243,False,-0.000045,-0.007767,0.007722,False,per_seed
1,pooled_H+h_vs_H+h+z_actual,-0.000148,-0.001246,0.000964,False,-0.001518,-0.005670,0.002719,False,-0.000081,-0.004253,0.003771,False,pooled
2,per_seed_H+h+z_actual_vs_H+h+z_random,-0.000641,-0.004288,0.002567,False,-0.004139,-0.013361,0.004994,False,-0.001540,-0.015936,0.010830,False,per_seed
3,pooled_H+h+z_actual_vs_H+h+z_random,-0.000556,-0.002111,0.000929,False,-0.003443,-0.008945,0.001872,False,-0.001183,-0.006816,0.004348,False,pooled


## §7 Ablations (trajectory path only)

**Optional** — isolates FFT and pooling on the **§3–§8 trajectory feature** \(z_{\mathrm{memory}}\). Does not affect Part I training or §D.1 deployment (\(z_j=M^j k\)).

| Ablation | Config | Feature map |
|----------|--------|-------------|
| `z_raw` | `use_fft=False` | \(z = W \,\mathrm{mean}_{\ell,j}(R)\) — time-domain trajectory |
| `z_no_attn` | `use_attention=False` | \(z = W \,\mathrm{mean}_{\ell,j}(S)\) — spectral mean-pool (default `ASSOC_CFG`) |

Compare deferral metrics to full config (`use_fft=True`, mean-pool). Learned attention (`use_attention=True`) requires a trained attention module and is **not** enabled here.

Primary paper probe uses **no FFT** (`z_j` magnitudes / `z_combined` in the probe notebook).

In [ ]:
# §7 — Trajectory-path ablations: raw R (no FFT) vs spectral mean-pool (ASSOC_CFG)

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §7 ablations")
    ablation_results = {}
else:
    ablation_results = {}
    for label, assoc in [
        ("z_raw", replace(ASSOC_CFG, use_fft=False)),
        ("z_no_attn", replace(ASSOC_CFG, use_attention=False)),
    ]:
        ab_cfg = replace(
            defer_cfg,
            output_dir=OUTPUT_DIR / f"deferral_ablation_{label}",
            associative_memory=assoc,
        )
        ablation_results[label] = run_msa_linear_deferral_experiment(ab_cfg)
    ablation_results.keys()

C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral.py:393: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cal_df[f"z_{j}"] = z[:, j]
C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral.py:662: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cal_df[f"z_{j}"] = z_cal[:, j]
C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\common\msa_linear_deferral

dict_keys(['z_raw', 'z_no_attn'])

## §8 Verdict draft (legacy downstream)

Aggregate §3–§7 numbers into `verdict_draft.json` for README / internal decision gates:

- **Reconstruction:** per-level MSE and cosine of \(\hat{v} = M^j k\) vs \(v = y - p\) (Part I)
- **Deferral (legacy):** pooled selective-risk for \(\tilde H\), \(\tilde H+h\), \(\tilde H+h+z_{\mathrm{memory}}\)
- **Incremental:** paired CI on \(\Delta\)SR(H+h → H+h+z)

**Paper-facing metrics** (AUROC/AUPRC for `z_combined` vs \(\tilde H\)) come from **`memory_vector_vs_magnitude_probe.ipynb`**, not this verdict file.

See `research/associative_memory_fft/README.md` for evidence-class criteria on the legacy downstream protocol.

In [ ]:
# §8 — Write verdict_draft.json summarizing reconstruction, deferral, incremental CI.

if not RUN_EVALUATION:
    print("RUN_EVALUATION=False — skipping §8 verdict draft")
else:
    verdict_payload = {
        "reconstruction": recon_summary.to_dict(),
        "deferral": defer_results.get("pooled_metrics") if defer_results else None,
        "incremental": (
            incr_results.get("paired_ci").to_dict()
            if incr_results and hasattr(incr_results.get("paired_ci"), "to_dict")
            else (incr_results.get("paired_ci") if incr_results else None)
        ),
    }
    with open(OUTPUT_DIR / "verdict_draft.json", "w", encoding="utf-8") as f:
        json.dump(verdict_payload, f, indent=2, default=str)
    print("Wrote", OUTPUT_DIR / "verdict_draft.json")


Wrote C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\artifacts\verdict_draft.json
